# LAS: Hough normal estimation and Open3D visualization

This notebook estimates a local Hough normal for every point in `data/point_cloud.las` and writes `outputs/point_cloud_hough_normals_k30.las`, with `normal_x`, `normal_y`, and `normal_z` stored as LAS Extra Bytes. It also writes a JSON sidecar with parameter, timing, and normal-quality information. Point order and the source LAS integer coordinates are preserved.

The method uses local KNN neighborhoods and distance-weighted orthogonality-consistency voting: `k=30`, `n_phi=32`, and `n_theta=64`, giving 512 candidate directions on the unit sphere. The normals are **unoriented axial results**: `n` and `-n` are equivalent. Apply a separate orientation step if consistent signs are needed.

Required input: a LAS point cloud at `data/point_cloud.las`, relative to the notebook working directory. Dependencies: Python 3.10 or later, `laspy`, `numpy`, `scipy`, and `open3d`. The estimator is implemented entirely in this notebook and needs no external Hough executable. The output directory is created automatically. A desktop OpenGL environment is required for the optional Open3D viewer.

> The supplied source excerpt did not include its `fibonacci_sphere` implementation. This notebook uses a standard Fibonacci sphere with equal-area centered sampling. If another source notebook uses a different helper, replace this function when bitwise reproduction of that implementation is required.


## 1. Configuration

The source notebook reports that `BATCH_SIZE=1000` was faster than 2,000 and 5,000 in a local 100,000-point benchmark; the benchmark evidence is not included here. Batch size does not change the KNN neighborhoods, candidate directions, or voting formula. Open3D subtracts `vis_origin` only for display; the Hough calculation and LAS export use the original coordinates.

`RECOMPUTE_NORMALS=True` recomputes normals and overwrites the output LAS and JSON sidecar. `RUN_VISUALIZATION=True` opens an interactive Open3D window in the final visualization cell; close the window to let that cell finish, or set the flag to `False` before a headless run. All points are displayed; only the normal arrows are sampled. The default arrow length is 50 times the estimated point spacing.


In [ ]:
from pathlib import Path
import json
import time

import laspy
import numpy as np
import open3d as o3d
from scipy.spatial import cKDTree

INPUT_LAS = Path("data/point_cloud.las")
OUTPUT_DIR = Path("outputs")
OUTPUT_LAS = OUTPUT_DIR / "point_cloud_hough_normals_k30.las"

K_NEIGHBORS = 30
N_PHI = 32
N_THETA = 64
SIGMA_DIST_FACTOR = 2.0
BATCH_SIZE = 1000

# True: recompute and overwrite OUTPUT_LAS; False: load normals from an existing output.
RECOMPUTE_NORMALS = True

# Open3D opens an interactive window. Set False for a headless Restart & Run All.
RUN_VISUALIZATION = True
NORMAL_ARROW_COUNT = 10_000  # Sample only arrows; keep all points in the viewer and output LAS.
NORMAL_ARROW_LENGTH = None   # None = 50 times the estimated point spacing.

if not INPUT_LAS.is_file():
    raise FileNotFoundError(f"Input LAS file not found: {INPUT_LAS}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Input: {INPUT_LAS}")
print(f"Output: {OUTPUT_LAS}")

## 2. Hough normal estimation functions


In [ ]:
def fibonacci_sphere(n: int) -> np.ndarray:
    """Return n nearly uniform unit-sphere directions using equal-area centered sampling."""
    indices = np.arange(n, dtype=np.float32) + np.float32(0.5)
    z = np.float32(1.0) - np.float32(2.0) * indices / np.float32(n)
    radius = np.sqrt(np.maximum(np.float32(0.0), np.float32(1.0) - z * z))
    golden_angle = np.float32(np.pi * (3.0 - np.sqrt(5.0)))
    azimuth = golden_angle * indices
    return np.column_stack((np.cos(azimuth) * radius, np.sin(azimuth) * radius, z)).astype(
        np.float32, copy=False
    )


def build_sphere_grid(n_phi: int = 32, n_theta: int = 64) -> np.ndarray:
    # As in the supplied code, the grid uses 32 * 64 // 4 = 512 candidate directions.
    n = max(n_phi * n_theta // 4, 64)
    return fibonacci_sphere(n)


def compute_normals_hough(
    points: np.ndarray,
    k: int = 30,
    n_phi: int = 32,
    n_theta: int = 64,
    batch_size: int = 1000,
    sigma_dist_factor: float = 2.0,
    tree: cKDTree | None = None,
) -> tuple[np.ndarray, dict]:
    """Stream KNN and orthogonality-consistency voting on the CPU without a full N-by-K array."""
    points = np.asarray(points, dtype=np.float32)
    n_points = len(points)
    actual_k = min(k + 1, n_points)  # The first nearest neighbor is the query point itself.
    if actual_k <= 1:
        raise RuntimeError(f"No valid neighbors (point count={n_points}, requested k={k}).")

    t_total = time.time()
    t_tree = t_knn = t_vote = 0.0
    t0 = time.time()
    if tree is None:
        tree = cKDTree(points)
    t_tree += time.time() - t0

    sphere_dirs = build_sphere_grid(n_phi=n_phi, n_theta=n_theta)
    sample_count = min(50_000, n_points)
    t0 = time.time()
    sample_distances, _ = tree.query(points[:sample_count], k=actual_k, workers=-1)
    t_knn += time.time() - t0
    if sample_distances.ndim == 1:
        sample_distances = sample_distances[:, None]
    spacing = float(np.median(sample_distances[:, min(3, sample_distances.shape[1] - 1)]))
    sigma_d = max(spacing * sigma_dist_factor, 1e-6)
    del sample_distances

    normals = np.empty((n_points, 3), dtype=np.float32)
    for start in range(0, n_points, batch_size):
        stop = min(start + batch_size, n_points)
        t0 = time.time()
        distances, indices = tree.query(points[start:stop], k=actual_k, workers=-1)
        t_knn += time.time() - t0
        if distances.ndim == 1:
            distances = distances[:, None]
            indices = indices[:, None]
        neighbor_indices = np.ascontiguousarray(indices[:, 1:actual_k], dtype=np.int32)
        np.clip(neighbor_indices, 0, n_points - 1, out=neighbor_indices)
        del distances, indices

        t0 = time.time()
        p = points[start:stop]
        q = points[neighbor_indices]
        u = q - p[:, None, :]
        u_norm = np.linalg.norm(u, axis=2, keepdims=True)
        u_hat = u / np.maximum(u_norm, 1e-10)
        distance = np.linalg.norm(u, axis=2)
        distance_weight = np.exp(-(distance**2) / (2.0 * sigma_d**2)).astype(np.float32)
        dots = np.einsum("bkd,md->bkm", u_hat, sphere_dirs)
        # Equivalent to the original sum(w * (1 - dot**2)), reusing dots in place
        # to avoid keeping both orthogonality scores and weighted B-by-K-by-M arrays.
        np.square(dots, out=dots)
        np.subtract(1.0, dots, out=dots)
        np.multiply(distance_weight[:, :, None], dots, out=dots)
        score = np.sum(dots, axis=1)
        normals[start:stop] = sphere_dirs[np.argmax(score, axis=1)]
        t_vote += time.time() - t0
        del neighbor_indices, p, q, u, u_norm, u_hat, distance, distance_weight
        del dots, score

        if stop % 50_000 == 0 or stop == n_points:
            print(f"  Processed {stop:,}/{n_points:,} points in {time.time() - t_total:.1f} s", flush=True)

    timing = {
        "total_seconds": time.time() - t_total,
        "tree_build_seconds": t_tree,
        "knn_query_seconds": t_knn,
        "vote_seconds": t_vote,
        "backend": "cpu",
        "n_points": n_points,
        "k": k,
        "n_phi": n_phi,
        "n_theta": n_theta,
        "candidate_directions": int(len(sphere_dirs)),
        "batch_size": batch_size,
        "sigma_dist_factor": sigma_dist_factor,
        "spacing": spacing,
        "sigma_d": sigma_d,
    }
    print(
        f"[hough breakdown] total={timing['total_seconds']:.2f}s "
        f"(tree={t_tree:.2f}s, knn={t_knn:.2f}s, vote={t_vote:.2f}s)"
    )
    return normals, timing

## 3. Load the point cloud and compute or load normals


In [ ]:
source_las = laspy.read(INPUT_LAS)
points = np.column_stack((source_las.x, source_las.y, source_las.z)).astype(np.float32, copy=False)
vis_origin = points.mean(axis=0)
print(f"Loaded {len(points):,} points; local rendering origin: {vis_origin}")

if RECOMPUTE_NORMALS or not OUTPUT_LAS.is_file():
    normals, hough_info = compute_normals_hough(
        points,
        k=K_NEIGHBORS,
        n_phi=N_PHI,
        n_theta=N_THETA,
        batch_size=BATCH_SIZE,
        sigma_dist_factor=SIGMA_DIST_FACTOR,
    )
else:
    cached_las = laspy.read(OUTPUT_LAS)
    required = {"normal_x", "normal_y", "normal_z"}
    available = set(cached_las.point_format.extra_dimension_names)
    if len(cached_las) != len(points) or not required.issubset(available):
        raise ValueError("The existing output has a different point count or missing normal fields; set RECOMPUTE_NORMALS to True.")
    normals = np.column_stack((cached_las.normal_x, cached_las.normal_y, cached_las.normal_z)).astype(
        np.float32, copy=False
    )
    hough_info = {"backend": "cached", "n_points": len(points)}
    print(f"Loaded existing normals from {OUTPUT_LAS}.")

normal_lengths = np.linalg.norm(normals, axis=1)
if not np.isfinite(normals).all() or not np.allclose(normal_lengths, 1.0, atol=1e-5):
    raise ValueError("All normals must be finite unit vectors.")
print(f"Normal length range: {normal_lengths.min():.8f}-{normal_lengths.max():.8f}")

## 4. Write normals to LAS Extra Bytes and verify the saved file


In [ ]:
if RECOMPUTE_NORMALS or not OUTPUT_LAS.is_file():
    output_las = laspy.read(INPUT_LAS)  # Preserve the source file and original point records.
    output_las.add_extra_dims([
        laspy.ExtraBytesParams(name="normal_x", type=np.float32, description="Hough local normal x"),
        laspy.ExtraBytesParams(name="normal_y", type=np.float32, description="Hough local normal y"),
        laspy.ExtraBytesParams(name="normal_z", type=np.float32, description="Hough local normal z"),
    ])
    output_las.normal_x = normals[:, 0]
    output_las.normal_y = normals[:, 1]
    output_las.normal_z = normals[:, 2]
    output_las.write(OUTPUT_LAS)

    sidecar_path = OUTPUT_LAS.with_suffix(".json")
    sidecar_path.write_text(json.dumps({
        "input_las": str(INPUT_LAS),
        "output_las": str(OUTPUT_LAS),
        "normal_sign": "unoriented axial normal; n and -n are equivalent",
        "finite_normal_fraction": float(np.isfinite(normals).all(axis=1).mean()),
        "normal_length_min": float(normal_lengths.min()),
        "normal_length_max": float(normal_lengths.max()),
        "timing": hough_info,
    }, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Written: {OUTPUT_LAS}")

checked_las = laspy.read(OUTPUT_LAS)
checked_normals = np.column_stack((checked_las.normal_x, checked_las.normal_y, checked_las.normal_z))
coordinates_unchanged = (
    len(checked_las) == len(source_las)
    and np.array_equal(checked_las.X, source_las.X)
    and np.array_equal(checked_las.Y, source_las.Y)
    and np.array_equal(checked_las.Z, source_las.Z)
)
print("LAS Extra Bytes:", list(checked_las.point_format.extra_dimension_names))
print("Point count and integer coordinates unchanged:", coordinates_unchanged)
print("All reloaded normals are finite:", bool(np.isfinite(checked_normals).all()))
if not coordinates_unchanged:
    raise RuntimeError("The output LAS point count or integer coordinates have changed.")

## 5. Open3D point cloud and normal visualization

The viewer displays the full point cloud. Normal arrows are sampled for visualization to keep the LineSet responsive. Rendering uses `points - vis_origin`; computation and export retain the original coordinates.


In [ ]:
def show_point_cloud_with_normals(
    points: np.ndarray,
    normals: np.ndarray,
    vis_origin: np.ndarray,
    spacing: float,
    arrow_count: int = 10_000,
    arrow_length: float | None = None,
) -> None:
    """Display the full point cloud and evenly sampled normal arrows."""
    local_points = np.asarray(points, dtype=np.float64) - np.asarray(vis_origin, dtype=np.float64)
    cloud = o3d.geometry.PointCloud()
    cloud.points = o3d.utility.Vector3dVector(local_points)
    cloud.normals = o3d.utility.Vector3dVector(np.asarray(normals, dtype=np.float64))
    cloud.paint_uniform_color([0.70, 0.72, 0.75])

    arrow_count = min(max(1, int(arrow_count)), len(local_points))
    arrow_indices = np.linspace(0, len(local_points) - 1, arrow_count, dtype=np.int64)
    if arrow_length is None:
        arrow_length = 50.0 * spacing
    starts = local_points[arrow_indices]
    ends = starts + float(arrow_length) * np.asarray(normals[arrow_indices], dtype=np.float64)
    arrows = o3d.geometry.LineSet()
    arrows.points = o3d.utility.Vector3dVector(np.vstack((starts, ends)))
    arrows.lines = o3d.utility.Vector2iVector(
        np.column_stack((np.arange(arrow_count), np.arange(arrow_count) + arrow_count))
    )
    arrows.colors = o3d.utility.Vector3dVector(np.tile([0.95, 0.15, 0.10], (arrow_count, 1)))

    frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=10.0 * float(arrow_length))
    print(
        f"Displaying all {len(local_points):,} points and {arrow_count:,} normal arrows; "
        f"arrow length = {arrow_length:.6g} (in the LAS coordinate units)."
    )
    o3d.visualization.draw_geometries(
        [cloud, arrows, frame],
        window_name="Open-source LAS: Hough normals",
        width=1600,
        height=900,
    )


if RUN_VISUALIZATION:
    visual_spacing = float(hough_info.get("spacing", 0.0))
    if visual_spacing <= 0:
        visual_spacing = float(np.median(cKDTree(points).query(points[:min(50_000, len(points))], k=4)[0][:, 3]))
    show_point_cloud_with_normals(
        points,
        normals,
        vis_origin,
        spacing=visual_spacing,
        arrow_count=NORMAL_ARROW_COUNT,
        arrow_length=NORMAL_ARROW_LENGTH,
    )
else:
    print("Visualization is disabled: set RUN_VISUALIZATION to True and rerun this cell.")